In [1]:
import os
for folder in ["src", "tests"]:
    os.makedirs(folder, exist_ok=True)

open("src/__init__.py", "w").close()
open("tests/__init__.py", "w").close()

In [2]:
%%writefile src/data.py

Writing src/data.py


In [3]:
%%writefile src/model.py

Writing src/model.py


In [4]:
%%writefile tests/test_pipeline_sanity.py

Writing tests/test_pipeline_sanity.py


In [5]:
!pip install pytest -q

In [6]:
!pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.5.2, langsmith-0.10.2, anyio-4.14.2
collected 0 items                                                              

============================ no tests ran in 0.02s =============================


In [7]:
import os
print("Current directory:", os.getcwd())
print("Files in tests/:", os.listdir("tests") if os.path.exists("tests") else "tests/ folder doesn't exist")

Current directory: /content
Files in tests/: ['__init__.py', '__pycache__', 'test_pipeline_sanity.py']


In [8]:
with open("tests/test_pipeline_sanity.py") as f:
    content = f.read()
print(repr(content[:200]))
print("---")
print("Total length:", len(content))

' '
---
Total length: 1


In [9]:
import shutil
shutil.rmtree("tests/__pycache__", ignore_errors=True)
shutil.rmtree(".pytest_cache", ignore_errors=True)

!pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.5.2, langsmith-0.10.2, anyio-4.14.2
collected 0 items                                                              

============================ no tests ran in 0.02s =============================


In [10]:
import os
for folder in ["src", "tests"]:
    os.makedirs(folder, exist_ok=True)
open("src/__init__.py", "w").close()
open("tests/__init__.py", "w").close()

In [11]:
%%writefile tests/test_pipeline_sanity.py
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.data import clean, tiny_clean
from src.model import build_model, train


def test_clean_produces_valid_schema():
    import pandas as pd
    import numpy as np

    raw = pd.DataFrame({
        "esi": [1, 2, 3, 4, 5, 99],
        "gender": ["Female", "Male", "Female", "Male", "Female", "Male"],
        "triage_vital_hr": [80, np.nan, 90, 75, 88, 95],
    })

    df = clean(raw)

    assert df["esi"].isin([1, 2, 3, 4, 5]).all()
    assert df["triage_vital_hr"].isna().sum() == 0
    assert set(df["gender"].dropna().unique()) <= {0, 1}


def test_smoke_train_predict():
    from sklearn.model_selection import train_test_split

    X, y = tiny_clean(60)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    model = build_model("random_forest", {"n_estimators": 20, "random_state": 42})
    train(model, X_train, y_train)
    preds = model.predict(X_test)

    assert len(preds) == len(y_test)

Overwriting tests/test_pipeline_sanity.py


In [12]:
with open("tests/test_pipeline_sanity.py") as f:
    content = f.read()
print("Length:", len(content))
print(content[:200])

Length: 1052
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

from src.data import clean, tiny_clean
from src.model import build_model, train


def test_clean_pr


In [13]:
import os
for fname in ["src/data.py", "src/model.py"]:
    if os.path.exists(fname):
        with open(fname) as f:
            length = len(f.read())
        print(f"{fname}: {length} characters")
    else:
        print(f"{fname}: MISSING")

src/data.py: 1 characters
src/model.py: 1 characters


In [14]:
%%writefile src/data.py
from pathlib import Path
import pandas as pd
import numpy as np

LEAKAGE_COLS = ["disposition", "previousdispo"]

VITALS = [
    "triage_vital_hr", "triage_vital_sbp", "triage_vital_dbp", "triage_vital_rr",
    "triage_vital_o2", "triage_vital_temp", "triage_glucose",
]

TARGET = "esi"


def load(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Could not find {path}.")
    return pd.read_csv(path, index_col=0)


def clean(raw):
    df = raw.copy()
    df = df[df[TARGET].isin([1, 2, 3, 4, 5])].copy()

    for col in VITALS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            median = df[col].median()
            df[col] = df[col].fillna(median)

    if "gender" in df.columns:
        mapped = df["gender"].map({"Female": 0, "Male": 1, 0: 0, 1: 1})
        if mapped.notna().sum() > 0:
            df["gender"] = mapped.astype("Int64")

    return df


def tiny_clean(n_rows=60):
    rng = np.random.default_rng(42)
    n = n_rows
    df = pd.DataFrame({
        "esi": rng.choice([1, 2, 3, 4, 5], size=n, p=[0.02, 0.3, 0.45, 0.18, 0.05]),
        "gender": rng.choice([0, 1], size=n),
        "triage_vital_hr": rng.normal(85, 15, size=n),
        "triage_vital_sbp": rng.normal(130, 20, size=n),
        "triage_vital_dbp": rng.normal(78, 12, size=n),
        "triage_vital_rr": rng.normal(18, 3, size=n),
        "triage_vital_o2": rng.normal(97, 2, size=n),
        "triage_vital_temp": rng.normal(98.2, 0.8, size=n),
        "triage_glucose": rng.normal(110, 30, size=n),
        "cc_abdominalpain": rng.choice([0, 1], size=n, p=[0.85, 0.15]),
        "cc_chestpain": rng.choice([0, 1], size=n, p=[0.9, 0.1]),
    })
    X = df.drop(columns=["esi"])
    y = df["esi"]
    return X, y

Overwriting src/data.py


In [15]:
%%writefile src/model.py
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report,
)

MODEL_REGISTRY = {
    "dummy": lambda params: DummyClassifier(**params),
    "logistic_regression": lambda params: LogisticRegression(**params),
    "decision_tree": lambda params: DecisionTreeClassifier(**params),
    "random_forest": lambda params: RandomForestClassifier(**params),
}


def build_model(model_name, params):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model '{model_name}'. Valid options: {list(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[model_name](params)


def train(model, X_train, y_train):
    model.fit(X_train, y_train)
    return model


def evaluate(model, X_test, y_test, esi1_label=1):
    preds = model.predict(X_test)
    return {
        "accuracy": round(accuracy_score(y_test, preds), 3),
        "precision_macro": round(precision_score(y_test, preds, average="macro", zero_division=0), 3),
        "recall_macro": round(recall_score(y_test, preds, average="macro", zero_division=0), 3),
        "f1_macro": round(f1_score(y_test, preds, average="macro"), 3),
        "recall_esi1": round(
            recall_score(y_test, preds, labels=[esi1_label], average=None, zero_division=0)[0], 3
        ),
    }


def classification_report_text(model, X_test, y_test):
    preds = model.predict(X_test)
    return classification_report(y_test, preds, digits=3, zero_division=0)

Overwriting src/model.py


In [16]:
!pytest tests/ -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.5.2, langsmith-0.10.2, anyio-4.14.2
collected 2 items                                                              

tests/test_pipeline_sanity.py::test_clean_produces_valid_schema PASSED   [ 50%]
tests/test_pipeline_sanity.py::test_smoke_train_predict PASSED           [100%]

============================== 2 passed in 4.79s ===============================
